In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os
from src.config import Configuration

CONFIG = Configuration()

In [ ]:
CHECKPOINTS = [
    # (path, use_radiomics, label)
    ("../models/Upenn-No_masks-Radiomics_2026-05-24-18-36-57/survival_checkpoint_best.ckpt",
     False, "No-Mask Rad"),
    ("../models/Upenn-Train_masks-Radiomics_2026-05-24-18-39-19/survival_checkpoint_best.ckpt",
     False, "Train-Mask Rad"),
]

for path, _, label in CHECKPOINTS:
    if not os.path.exists(path):
        print(f"MISSING: {path}")
    else:
        print(f"  OK: {path}")

In [ ]:
import copy
import os

import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from tqdm.notebook import tqdm

from src.data.UPennGBMDataset import UPennGBMDataset
from src.models.survival_predictor import MultimodalSurvivalPredictor
from maikol_utils.print_utils import print_color

REGIONS = ["ED", "ET", "NC"]
CHANNELS = ["canal_T1", "canal_T1GD", "canal_T2", "canal_FLAIR"]
RADIOMIC_REGIONS = ["ED", "ET", "NC"]
TABULAR_FEATURES = ["Gender", "Age", "KPS", "IDH1", "MGMT", "GTR"]

IMAGE_CHANNEL_IDX = {"canal_T1": 0, "canal_T1GD": 1, "canal_T2": 2, "canal_FLAIR": 3}

TABULAR_COL_INDICES = {
    "Gender": [0],
    "Age": [1],
    "KPS": [2, 3, 4],
    "IDH1": [5, 6, 7],
    "MGMT": [8, 9, 10],
    "GTR": [11, 12, 13],
}

BASE_TEMPLATE = {
    "canal_T1": True, "canal_T1GD": True, "canal_T2": True, "canal_FLAIR": True,
    "RADIOMIC": {"ET": True, "ED": True, "NC": True},
    "TABULAR": {"Gender": True, "Age": True, "KPS": True,
                 "IDH1": True, "MGMT": True, "GTR": True},
}

In [ ]:
def generate_test_configs():
    configs = {}
    configs["base"] = copy.deepcopy(BASE_TEMPLATE)

    for ch in CHANNELS:
        cfg = copy.deepcopy(BASE_TEMPLATE)
        cfg[ch] = False
        configs[f"no_{ch}"] = cfg

    for region in RADIOMIC_REGIONS:
        cfg = copy.deepcopy(BASE_TEMPLATE)
        cfg["RADIOMIC"][region] = False
        configs[f"no_radiomic_{region}"] = cfg

    for feat in TABULAR_FEATURES:
        cfg = copy.deepcopy(BASE_TEMPLATE)
        cfg["TABULAR"][feat] = False
        configs[f"no_tabular_{feat}"] = cfg

    cfg = copy.deepcopy(BASE_TEMPLATE)
    for ch in CHANNELS:
        cfg[ch] = False
    configs["no_all_channels"] = cfg

    cfg = copy.deepcopy(BASE_TEMPLATE)
    for region in RADIOMIC_REGIONS:
        cfg["RADIOMIC"][region] = False
    configs["no_all_radiomic"] = cfg

    cfg = copy.deepcopy(BASE_TEMPLATE)
    for feat in TABULAR_FEATURES:
        cfg["TABULAR"][feat] = False
    configs["no_all_tabular"] = cfg

    return configs


def apply_config_to_batch(batch, config, device):
    image = batch['image'].to(device)
    tabular = batch['tabular'].to(device)
    labels = batch['label'].to(device)
    B = image.shape[0]

    image_mask = torch.zeros(B, len(CHANNELS), device=device)
    for ch_name, ch_idx in IMAGE_CHANNEL_IDX.items():
        if config[ch_name]:
            image_mask[:, ch_idx] = 1.0
        else:
            image[:, ch_idx] = 0.0

    radiomic = {
        'ED': batch['radiomic_ED'].to(device),
        'ET': batch['radiomic_ET'].to(device),
        'NC': batch['radiomic_NC'].to(device),
    }
    radiomic_mask = {}
    for region in REGIONS:
        radiomic_mask[region] = torch.ones(B, 4, device=device) if config["RADIOMIC"][region] \
            else torch.zeros(B, 4, device=device)

    tabular_mask = torch.zeros(B, 14, device=device)
    for feat in TABULAR_FEATURES:
        if config["TABULAR"][feat]:
            for col_idx in TABULAR_COL_INDICES[feat]:
                tabular_mask[:, col_idx] = 1.0
        else:
            for col_idx in TABULAR_COL_INDICES[feat]:
                tabular[:, col_idx] = 0.0

    return image, tabular, labels, radiomic, radiomic_mask, tabular_mask, image_mask

In [ ]:
def run_ablation(model, test_loader, device):
    model.eval()
    configs = generate_test_configs()
    results = {}

    with torch.no_grad():
        for cfg_name, cfg in tqdm(configs.items(), desc="Configs"):
            all_preds, all_labels = [], []

            for batch in tqdm(test_loader, desc=f"  [{cfg_name}]", leave=False):
                image, tabular, labels, radiomic, radiomic_mask, tabular_mask, image_mask = \
                    apply_config_to_batch(batch, cfg, device)

                preds = model(image, tabular, radiomic, radiomic_mask,
                              tabular_mask=tabular_mask,
                              image_mask=image_mask)
                all_preds.extend(torch.argmax(preds, dim=1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

            results[cfg_name] = {
                'f1': float(f1_score(all_labels, all_preds, average='weighted')),
                'precision': float(precision_score(all_labels, all_preds, average='weighted', zero_division=0)),
                'recall': float(recall_score(all_labels, all_preds, average='weighted', zero_division=0)),
                'accuracy': float(accuracy_score(all_labels, all_preds)),
            }

    return results

In [ ]:
def load_model_for_ablation(checkpoint_path, use_radiomics, CONFIG):
    model = MultimodalSurvivalPredictor(
        num_classes=len(CONFIG.labels),
        embed_dim=CONFIG.ssl_embed_dim,
        num_heads=CONFIG.ssl_num_heads,
        dropout=CONFIG.ssl_dropout,
        in_channels=4,
        patch_size=CONFIG.ssl_patch_size,
        vit_depth=CONFIG.ssl_vit_depth,
        vol_size=CONFIG.ssl_vol_size,
        pos_embed=CONFIG.pos_embed,
        use_radiomics=use_radiomics,
    )

    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint))
    model.load_state_dict(state_dict, strict=False)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {use_radiomics=}, params={n_params:,}")

    return model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

test_dataset = UPennGBMDataset(
    CONFIG=CONFIG,
    partition='test',
    apply_mask=False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG.survival_batch_size,
    shuffle=False,
    num_workers=CONFIG.survival_num_workers,
)

all_results = {}

for ckpt_path, use_radiomics, label in CHECKPOINTS:
    print_color(f"\n{'='*60}", 'cyan')
    print_color(f"  {label}", 'cyan')
    print_color(f"  {ckpt_path}", 'cyan')
    print_color(f"{'='*60}", 'cyan')

    model = load_model_for_ablation(ckpt_path, use_radiomics, CONFIG)
    model = model.to(device)

    results = run_ablation(model, test_loader, device)
    all_results[label] = results

    del model
    torch.cuda.empty_cache()

In [ ]:
for label, results in all_results.items():
    print()
    print_color(f"  {label}", 'cyan')
    print("-" * 80)
    header = f"  {'Config':<28} {'F1':>8} {'Precision':>10} {'Recall':>8} {'Accuracy':>10}"
    print(header)
    print("  " + "-" * 76)
    for cfg_name, res in results.items():
        marker = "  <-- baseline" if cfg_name == "base" else ""
        print(f"  {cfg_name:<28} {res['f1']:>8.4f} {res['precision']:>10.4f} {res['recall']:>8.4f} {res['accuracy']:>10.4f} {marker}")
    print()

In [ ]:
import matplotlib.pyplot as plt

config_names = list(next(iter(all_results.values())).keys())
x = range(len(config_names))

n_models = len(all_results)
fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 5), squeeze=False)

for ax, (label, results) in zip(axes[0], all_results.items()):
    ax.bar(x, [results[c]['accuracy'] for c in config_names], width=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(config_names, rotation=45, ha='right', fontsize=7)
    ax.set_ylim(0, 1)
    ax.set_title(f"{label}")
    ax.set_ylabel('Accuracy')
    ax.axhline(y=results['base']['accuracy'], color='red', linestyle='--', linewidth=1)

fig.suptitle('Accuracy by Missing-Data Configuration', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

config_names = list(next(iter(all_results.values())).keys())
x = range(len(config_names))
width = 0.8 / len(all_results)

fig, ax = plt.subplots(figsize=(16, 5))

for i, (label, results) in enumerate(all_results.items()):
    offset = (i - len(all_results) / 2 + 0.5) * width
    ax.bar([xi + offset for xi in x],
           [results[c]['f1'] for c in config_names],
           width, label=label)

ax.set_xticks(x)
ax.set_xticklabels(config_names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1)
ax.set_title('F1 Score Comparison Across Models')
ax.legend()
plt.tight_layout()
plt.show()